# FormalBench EF-DA-TASK: Lean 4 AGI Measurement Benchmark

## Executive Functions (EF) Track - Directing Action (DA) Sub-Abilities

This notebook evaluates Lean 4 proof generation on the **vSPACE v_train_extended.csv** dataset, measuring Executive Functions cognitive abilities as defined in the DeepMind AGI cognitive framework.

### Cognitive Framework Reference

Based on: *Measuring Progress Toward AGI: A Cognitive Framework* (DeepMind, 2025)
https://storage.googleapis.com/deepmind-media/DeepMind.com/Blog/measuring-progress-toward-agi/measuring-progress-toward-agi-a-cognitive-framework.pdf

**EF-DA-TASK Taxonomy (§7.8)**:
- **§7.8.1 Planning**: Ability to formulate sequences of future actions to achieve specific goals (Owen, 1997)
  - Building and pruning decision trees (Mattar and Lengyel, 2022)
  - Multi-step problem solving
  - Long-term goal achievement
- **§7.8.2 Goal Setting/Maintenance**: Ability to set and maintain goals to organize and direct action (Buschman and Miller, 2014; Dickinson and Balleine, 1994)
  - Goal hierarchy management
  - Working memory for goal states
  - Persistence through distractions

### Dataset: vSPACE v_train_extended.csv

- **Total proof tasks**: 1,239
- **Difficulty categories**: 6 (basic, advanced, coq_based, basic_core, autonomous, augmented)
- **Domain**: Formal verification of cryptographic protocols, election systems, zero-knowledge proofs
- **Format**: Lean 4 proof tasks with `-- !benchmark @start/@end` markers

### Benchmark Structure

This unified notebook will be split into 6 task-specific notebooks:
1. `01_Planning_MultiStep_Proofs.ipynb` - Planning ability with multi-step Lean 4 proofs
2. `02_Goal_Setting_Theorem_Statements.ipynb` - Goal setting/maintenance for theorem formulation
3. `03_Planning_Cryptographic_Proofs.ipynb` - Planning for cryptographic protocol verification
4. `04_Goal_Setting_Proof_Strategies.ipynb` - Goal maintenance through proof strategy selection
5. `05_Planning_Decision_Tree_Search.ipynb` - Decision tree construction for tactic selection
6. `06_Goal_Setting_Working_Memory.ipynb` - Working memory for proof state management

### Kaggle Benchmarks SDK

Reference: https://github.com/Kaggle/kaggle-benchmarks/blob/ci/cookbook.md

```python
import kaggle_benchmarks as kbench

@kbench.task(name="ef_da_task")
def ef_da_task(llm, proof_task) -> dict:
    response = llm.prompt(proof_task['lean_code'])
    return {
        'task_id': proof_task['id'],
        'difficulty': proof_task['difficulty'],
        'predicted_proof': response,
        'is_correct': verify_proof(response, proof_task)
    }
```

### Metrics

- **Proof Completion Rate**: % of proofs that compile successfully
- **Tactic Correctness**: % of tactics that advance proof state
- **Goal Achievement**: % of theorems fully proven
- **Planning Efficiency**: Average tactics per successful proof
- **Goal Maintenance**: % of proofs that don't lose track of subgoals

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Any, Tuple
import json
import hashlib

# Kaggle Benchmarks SDK
import kaggle_benchmarks as kbench

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
!curl -LO https://raw.githubusercontent.com/TheSDEs/temp/refs/heads/main/_references/secure-e-voting/v_train_extended.csv
CSV_PATH = Path('v_train_extended.csv')

df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(df)} proof tasks")
print(f"\nDifficulty distribution:")
print(df['difficulty'].value_counts())
print(f"\nColumns: {list(df.columns)}")

## EF-DA-TASK 1: Planning - Multi-Step Proof Construction

### Cognitive Ability
Planning involves formulating sequences of future actions (tactics) to achieve specific goals (theorem proofs). This requires:
- **Decision tree construction**: Building possible tactic sequences
- **Forward planning**: Anticipating proof state changes
- **Backtracking**: Pruning failed branches

### Task Definition
Given a Lean 4 theorem statement with `sorry`, generate a complete proof using a sequence of tactics that:
1. Identifies the theorem structure
2. Selects appropriate tactics
3. Manages subgoals
4. Completes all branches

In [3]:
@kbench.task(name="planning_multistep_proof_task", store_task=True)
def planning_multistep_proof_task(llm, id: str, description: str, lean_code: str, difficulty: str) -> dict:
    """
    EF-DA-TASK Planning: Multi-Step Proof Construction
    
    Measures the ability to plan and execute a sequence of Lean 4 tactics
    to complete a formal proof.
    
    Args:
        llm: Language model interface
        id: Task identifier
        description: Natural language description of the proof goal
        lean_code: Lean 4 code with `sorry` placeholder
        difficulty: Difficulty category
    
    Returns:
        Dictionary with planning metrics
    """
    # Construct prompt with proof context
    prompt = f"""Complete the following Lean 4 proof:

{description}

```lean4
{lean_code}
```

Replace `sorry` with a complete formal proof. Use appropriate tactics for the difficulty level ({difficulty}).

Difficulty: {difficulty}
Task ID: {id}"""
    
    # Generate proof
    response = llm.prompt(prompt, max_tokens=2048, temperature=0.2)
    
    # Extract metrics
    tactics_used = count_tactics(response)
    proof_structure_valid = validate_proof_structure(response)
    subgoals_managed = count_subgoals_managed(response)
    
    return {
        'task_id': id,
        'difficulty': difficulty,
        'ef_da_subability': 'Planning',
        'ef_da_task': 'Multi-Step Proof Construction',
        'predicted_proof': response,
        'tactics_count': tactics_used,
        'proof_structure_valid': proof_structure_valid,
        'subgoals_managed': subgoals_managed,
        'planning_efficiency': calculate_planning_efficiency(tactics_used, difficulty),
        'tokens_used': len(response.split())
    }


def count_tactics(proof: str) -> int:
    """Count number of tactics in proof."""
    tactic_keywords = ['apply', 'intro', 'induction', 'cases', 'rw', 'simp', 'exact', 
                       'constructor', 'left', 'right', 'split', 'refine', 'use', 'have',
                       'assert', ' generalize', 'revert', 'clear', 'subst', 'specialize']
    count = 0
    for keyword in tactic_keywords:
        count += proof.lower().count(keyword)
    return count


def validate_proof_structure(proof: str) -> bool:
    """Check if proof has valid structure."""
    # Check for theorem/definition
    has_theorem = 'theorem' in proof or 'lemma' in proof or 'def' in proof
    # Check for proof block
    has_proof = ':=' in proof or 'by' in proof
    # Check no remaining sorries
    no_sorries = 'sorry' not in proof
    # Check balanced braces
    balanced = proof.count('{') == proof.count('}')
    
    return has_theorem and has_proof and no_sorries and balanced


def count_subgoals_managed(proof: str) -> int:
    """Count number of subgoals explicitly managed."""
    # Count case splits, constructor applications, etc.
    subgoal_markers = ['case', ' · ', 'next', 'all_goals', 'focus']
    count = 0
    for marker in subgoal_markers:
        count += proof.count(marker)
    return count


def calculate_planning_efficiency(tactics: int, difficulty: str) -> float:
    """Calculate planning efficiency score (0-1)."""
    # Expected tactics by difficulty
    expected = {
        'basic': 5,
        'advanced': 10,
        'coq_based': 15,
        'basic_core': 10,
        'autonomous': 20,
        'augmented': 15
    }
    
    exp = expected.get(difficulty, 10)
    # Efficiency = 1.0 if tactics match expected, decreases for over/under
    if tactics == 0:
        return 0.0
    ratio = tactics / exp
    if 0.5 <= ratio <= 2.0:
        return max(0.0, 1.0 - abs(1.0 - ratio))
    else:
        return 0.0

In [4]:
# Test the task with a sample
sample_row = df.iloc[0]

print(f"Testing with task: {sample_row['id']}")
print(f"Difficulty: {sample_row['difficulty']}")
print(f"\nDescription (first 200 chars):\n{sample_row['description'][:200]}...")
print(f"\nLean code (first 300 chars):\n{sample_row['lean_code'][:300]}...")

## EF-DA-TASK 2: Goal Setting/Maintenance - Theorem Statement Formulation

### Cognitive Ability
Goal Setting/Maintenance involves setting and maintaining goals to organize and direct action. In theorem proving:
- **Goal formulation**: Correctly stating what needs to be proven
- **Goal hierarchy**: Managing parent goals and subgoals
- **Working memory**: Keeping track of assumptions and proven facts

### Task Definition
Given a natural language description of a mathematical property, formulate the correct Lean 4 theorem statement with:
1. Correct type signatures
2. Appropriate quantifiers
3. Proper precondition/postcondition structure
4. Complete goal statement

In [5]:
@kbench.task(name="goal_setting_theorem_task", store_task=True)
def goal_setting_theorem_task(llm, id: str, description: str, difficulty: str) -> dict:
    """
    EF-DA-TASK Goal Setting/Maintenance: Theorem Statement Formulation
    
    Measures the ability to formulate correct theorem statements from
    natural language descriptions.
    
    Args:
        llm: Language model interface
        id: Task identifier
        description: Natural language description
        difficulty: Difficulty category
    
    Returns:
        Dictionary with goal-setting metrics
    """
    prompt = f"""Formulate a Lean 4 theorem statement for the following property:

{description}

The theorem should include:
1. Correct type signatures for all variables
2. Appropriate quantifiers (∀, ∃)
3. Preconditions (if any)
4. The main goal statement

Difficulty: {difficulty}
Task ID: {id}

Provide ONLY the theorem statement (no proof)."""
    
    response = llm.prompt(prompt, max_tokens=512, temperature=0.1)
    
    # Analyze goal formulation quality
    has_types = check_type_signatures(response)
    has_quantifiers = check_quantifiers(response, description)
    has_structure = check_theorem_structure(response)
    goal_clarity = measure_goal_clarity(response)
    
    return {
        'task_id': id,
        'difficulty': difficulty,
        'ef_da_subability': 'Goal Setting/Maintenance',
        'ef_da_task': 'Theorem Statement Formulation',
        'predicted_theorem': response,
        'has_type_signatures': has_types,
        'has_appropriate_quantifiers': has_quantifiers,
        'has_correct_structure': has_structure,
        'goal_clarity_score': goal_clarity,
        'goal_maintenance_score': calculate_goal_maintenance(has_types, has_quantifiers, has_structure)
    }


def check_type_signatures(theorem: str) -> bool:
    """Check if theorem has type signatures."""
    type_indicators = [': Type', ': ℕ', ': Prop', ': Bool', '→', '→']
    return any(ind in theorem for ind in type_indicators)


def check_quantifiers(theorem: str, description: str) -> bool:
    """Check if theorem has appropriate quantifiers."""
    # Check for universal quantification
    has_forall = '∀' in theorem or 'forall' in theorem.lower()
    # Check for existential if description implies it
    needs_exists = 'exists' in description.lower() or 'there exists' in description.lower()
    has_exists = '∃' in theorem or 'exists' in theorem.lower()
    
    if needs_exists:
        return has_exists
    return has_forall or True  # Allow implicit quantification


def check_theorem_structure(theorem: str) -> bool:
    """Check if theorem has correct structure."""
    has_keyword = 'theorem' in theorem.lower() or 'lemma' in theorem.lower()
    has_name = len(theorem.split()) > 2  # Should have name and body
    has_colon = ':' in theorem  # Type separator
    
    return has_keyword and has_name and has_colon


def measure_goal_clarity(theorem: str) -> float:
    """Measure clarity of goal statement (0-1)."""
    # Simple heuristic: longer, more specific theorems are clearer
    length = len(theorem.split())
    if length < 10:
        return 0.3
    elif length < 20:
        return 0.6
    elif length < 40:
        return 0.8
    else:
        return 1.0


def calculate_goal_maintenance(types: bool, quants: bool, structure: bool) -> float:
    """Calculate overall goal maintenance score."""
    score = sum([types, quants, structure]) / 3.0
    return score

## Evaluation Pipeline

### Running Full Benchmark

This section demonstrates how to run the complete EF-DA-TASK evaluation on the full dataset.

In [6]:
def run_ef_da_benchmark(df: pd.DataFrame, llm, n_tasks: int = 10) -> pd.DataFrame:
    """
    Run EF-DA-TASK benchmark on subset of data.
    
    Args:
        df: Full dataset
        llm: Language model
        n_tasks: Number of tasks to evaluate
    
    Returns:
        DataFrame with results
    """
    # Sample tasks across difficulties
    sampled = df.groupby('difficulty', group_keys=False).apply(
        lambda x: x.sample(n=min(n_tasks // 6, len(x)))
    )
    
    results = []
    
    print(f"Running EF-DA-TASK benchmark on {len(sampled)} tasks...")
    
    for idx, row in sampled.iterrows():
        # Run planning task
        planning_result = planning_multistep_proof_task.run(
            llm,
            id=row['id'],
            description=row['description'],
            lean_code=row['lean_code'],
            difficulty=row['difficulty']
        )
        results.append(planning_result)
        
        # Run goal setting task
        goal_result = goal_setting_theorem_task.run(
            llm,
            id=row['id'],
            description=row['description'],
            difficulty=row['difficulty']
        )
        results.append(goal_result)
        
        print(f"  Completed {row['id']} ({row['difficulty']})")
    
    return pd.DataFrame(results)


def analyze_results(results_df: pd.DataFrame) -> dict:
    """
    Analyze benchmark results.
    
    Args:
        results_df: Results DataFrame
    
    Returns:
        Dictionary with analysis metrics
    """
    analysis = {
        'total_tasks': len(results_df),
        'by_subability': {},
        'by_difficulty': {},
        'planning_metrics': {},
        'goal_metrics': {}
    }
    
    # Group by EF-DA subability
    for subability in results_df['ef_da_subability'].unique():
        subset = results_df[results_df['ef_da_subability'] == subability]
        analysis['by_subability'][subability] = {
            'count': len(subset),
            'mean_score': subset.get('planning_efficiency', subset.get('goal_maintenance_score', pd.Series())).mean()
        }
    
    # Group by difficulty
    for difficulty in results_df['difficulty'].unique():
        subset = results_df[results_df['difficulty'] == difficulty]
        analysis['by_difficulty'][difficulty] = {
            'count': len(subset),
            'mean_efficiency': subset.get('planning_efficiency', pd.Series([0])).mean()
        }
    
    # Planning-specific metrics
    planning_df = results_df[results_df['ef_da_subability'] == 'Planning']
    if len(planning_df) > 0:
        analysis['planning_metrics'] = {
            'avg_tactics': planning_df['tactics_count'].mean(),
            'structure_valid_rate': planning_df['proof_structure_valid'].mean(),
            'avg_subgoals': planning_df['subgoals_managed'].mean()
        }
    
    # Goal-specific metrics
    goal_df = results_df[results_df['ef_da_subability'] == 'Goal Setting/Maintenance']
    if len(goal_df) > 0:
        analysis['goal_metrics'] = {
            'type_signature_rate': goal_df['has_type_signatures'].mean(),
            'quantifier_rate': goal_df['has_appropriate_quantifiers'].mean(),
            'structure_rate': goal_df['has_correct_structure'].mean(),
            'avg_clarity': goal_df['goal_clarity_score'].mean()
        }
    
    return analysis

## Visualization

### Plot EF-DA-TASK Performance

In [7]:
def plot_ef_da_results(analysis: dict):
    """Create visualization of EF-DA-TASK results."""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Performance by subability
    ax1 = axes[0, 0]
    subabilities = list(analysis['by_subability'].keys())
    scores = [analysis['by_subability'][s]['mean_score'] for s in subabilities]
    ax1.bar(subabilities, scores, color=['#3498db', '#e74c3c'])
    ax1.set_title('EF-DA Subability Performance')
    ax1.set_ylabel('Mean Score')
    ax1.set_ylim(0, 1)
    
    # 2. Performance by difficulty
    ax2 = axes[0, 1]
    difficulties = list(analysis['by_difficulty'].keys())
    efficiencies = [analysis['by_difficulty'][d]['mean_efficiency'] for d in difficulties]
    ax2.bar(difficulties, efficiencies, color='#2ecc71')
    ax2.set_title('Planning Efficiency by Difficulty')
    ax2.set_ylabel('Mean Efficiency')
    ax2.set_ylim(0, 1)
    ax2.tick_params(axis='x', rotation=45)
    
    # 3. Planning metrics
    ax3 = axes[1, 0]
    if analysis['planning_metrics']:
        metrics = ['Avg Tactics', 'Structure Valid %', 'Avg Subgoals']
        values = [
            analysis['planning_metrics']['avg_tactics'],
            analysis['planning_metrics']['structure_valid_rate'] * 100,
            analysis['planning_metrics']['avg_subgoals']
        ]
        ax3.bar(metrics, values, color='#9b59b6')
        ax3.set_title('Planning Metrics')
    
    # 4. Goal metrics
    ax4 = axes[1, 1]
    if analysis['goal_metrics']:
        metrics = ['Type Sig %', 'Quantifier %', 'Structure %', 'Avg Clarity']
        values = [
            analysis['goal_metrics']['type_signature_rate'] * 100,
            analysis['goal_metrics']['quantifier_rate'] * 100,
            analysis['goal_metrics']['structure_rate'] * 100,
            analysis['goal_metrics']['avg_clarity']
        ]
        ax4.bar(metrics, values, color='#f39c12')
        ax4.set_title('Goal Setting/Maintenance Metrics')
        ax4.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('ef_da_task_results.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nVisualization saved to: ef_da_task_results.png")

## Example Run

### Mock LLM for Testing

In [8]:
class MockLLM:
    """Mock LLM for testing without API calls."""
    
    def prompt(self, prompt: str, max_tokens: int = 2048, temperature: float = 0.2) -> str:
        """Generate mock response."""
        if 'theorem statement' in prompt.lower():
            return """theorem mod_add_assoc (P : ℕ) (a b c : ZMod P) 
  (h_precond : Nat.Prime P): 
  (a + b) + c = a + (b + c) := by
  apply add_assoc"""
        else:
            return """theorem feature_proof (input : Type) (h_precond : proof_precond input) : Prop := by
  intro h
  apply h_precond
  simp
  exact h"""


# Test with mock LLM
mock_llm = MockLLM()

print("Testing with mock LLM...")
test_result = planning_multistep_proof_task.run(
    mock_llm,
    id=df.iloc[0]['id'],
    description=df.iloc[0]['description'],
    lean_code=df.iloc[0]['lean_code'],
    difficulty=df.iloc[0]['difficulty']
)

print(f"\nTest result keys: {list(test_result.keys())}")
print(f"Planning efficiency: {test_result.get('planning_efficiency', 'N/A')}")
print(f"Tactics count: {test_result.get('tactics_count', 'N/A')}")
print(f"Proof structure valid: {test_result.get('proof_structure_valid', 'N/A')}")

## Splitting into 6 Task Notebooks

This unified notebook should be split into 6 focused notebooks:

### 1. `01_Planning_MultiStep_Proofs.ipynb`
- **EF-DA Subability**: Planning
- **Task**: Multi-step proof construction
- **Metrics**: Tactics count, structure validity, subgoal management
- **Dataset**: All 1,239 tasks

### 2. `02_Goal_Setting_Theorem_Statements.ipynb`
- **EF-DA Subability**: Goal Setting/Maintenance
- **Task**: Theorem statement formulation
- **Metrics**: Type signatures, quantifiers, goal clarity
- **Dataset**: Subset with clear theorem statements

### 3. `03_Planning_Cryptographic_Proofs.ipynb`
- **EF-DA Subability**: Planning
- **Task**: Cryptographic protocol verification
- **Metrics**: Security property preservation, tactic sequences
- **Dataset**: coq_based, basic_core difficulties

### 4. `04_Goal_Setting_Proof_Strategies.ipynb`
- **EF-DA Subability**: Goal Setting/Maintenance
- **Task**: Proof strategy selection
- **Metrics**: Strategy appropriateness, goal maintenance
- **Dataset**: autonomous, augmented difficulties

### 5. `05_Planning_Decision_Tree_Search.ipynb`
- **EF-DA Subability**: Planning
- **Task**: Decision tree construction for tactic selection
- **Metrics**: Tree depth, branching factor, pruning efficiency
- **Dataset**: advanced, autonomous difficulties

### 6. `06_Goal_Setting_Working_Memory.ipynb`
- **EF-DA Subability**: Goal Setting/Maintenance
- **Task**: Working memory for proof state management
- **Metrics**: Subgoal tracking, assumption management
- **Dataset**: All difficulties with complex subgoals

## Summary

### EF-DA-TASK Benchmark Framework

This notebook provides a comprehensive framework for measuring Executive Functions (EF) cognitive abilities in Lean 4 proof generation, specifically:

1. **Planning** (§7.8.1): Multi-step proof construction, decision tree search, cryptographic verification
2. **Goal Setting/Maintenance** (§7.8.2): Theorem formulation, proof strategy selection, working memory

### Next Steps

1. Split this notebook into 6 task-specific notebooks
2. Integrate with actual Lean 4 proof checker for validation
3. Run on full dataset (1,239 tasks)
4. Submit to Kaggle Measuring AGI competition

### References

- Diamond, A. (2013). Executive functions. *Annual Review of Psychology*
- Owen, A. M. (1997). Cognitive planning in humans: Neuropsychological, neuroanatomical and neuropharmacological perspectives.
- Buschman, T. J., & Miller, E. K. (2014). Goal-direction and top-down control.
- Dickinson, A., & Balleine, B. (1994). Motivational control of goal-directed action.
- Mattar, M. G., & Lengyel, M. (2022). Planning as inference in a hierarchical model.
- DeepMind AGI Cognitive Framework: https://storage.googleapis.com/deepmind-media/DeepMind.com/Blog/measuring-progress-toward-agi/measuring-progress-toward-agi-a-cognitive-framework.pdf
- Kaggle Benchmarks SDK: https://github.com/Kaggle/kaggle-benchmarks